# Text2Cypher LoRA fine-tune — exploring schema-conditioned Cypher generation (RunPod)

A scratch notebook for fine-tuning **Qwen3-4B-Instruct** with Unsloth + LoRA on a synthetic
natural-language → Cypher dataset. Each row pairs a graph schema, a question, and a gold query;
schemas are grouped so train/validation/heldout splits never leak the same schema across splits.

I'm using this to experiment with:
- schema-grouped splits (80/10/10 by `schema_id`, not by row)
- a small effective batch (16) with cosine LR and early stopping on `eval_loss`
- greedy inference + strict exact-match scoring (plus optional semantic rescoring offline)

Expect to edit `BASE_DIR`, `DATASET_PATH`, and the Hugging Face repo id before running.


Points the notebook at the dataset and output locations. Paths are relative to the repo
root — run this notebook from there (or edit `BASE_DIR` to an absolute path). `OUTPUT_DIR` is
where checkpoints, the saved adapter, logs, and the report all get written.


In [ ]:
import os
os.chdir("/workspace/Text2Cypher")  # edit to your clone path
print(os.getcwd())


In [ ]:
from pathlib import Path

BASE_DIR = Path("public/dataset")  # change if you moved the dataset elsewhere
OUTPUT_DIR = Path("public/models/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = BASE_DIR / "synthetic_cypher_dataset.parquet"
assert DATASET_PATH.exists(), (
    f"{DATASET_PATH} not found — generate the synthetic dataset first (see "
    f"your dataset build script) before running this notebook."
)


Quick GPU check before we load anything — make sure a card is actually attached.

In [ ]:
!nvidia-smi

Confirms PyTorch can actually see that GPU.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


Standard imports used throughout the rest of the notebook.

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
import warnings
warnings.simplefilter("ignore")
from pprint import pprint


Sets up run-scoped logging — every notebook run gets its own timestamped log file under
`outputs/logs/`, plus a console stream, so training progress, evaluation progress, and any errors
are all captured somewhere durable instead of only living in cell output.


In [ ]:
import logging
from datetime import datetime

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = Path("public/logs")
EVAL_DIR = Path("public/evals")
LOG_DIR.mkdir(exist_ok=True)
LOG_PATH = LOG_DIR / f"run_{RUN_ID}.log"

logger = logging.getLogger("text2cypher_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_PATH)
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info(f"Run started | RUN_ID={RUN_ID}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)} | "
                f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
logger.info(f"PyTorch: {torch.__version__}")
logger.info(f"Log file for this run: {LOG_PATH}")


Loads the synthetic dataset and splits it **by schema, not by row**: shuffles the unique
`schema_id` values with a fixed seed, then assigns whole schemas to train (80%), validation (10%,
used only for `eval_loss`/early stopping during training), or heldout (10%, never touched during
training at all — the real generalization number). A random row split would let other questions from
the same schema leak across splits, which mostly tests memorization instead.


In [ ]:
RNG_SEED = 42

raw_df = pd.read_parquet(DATASET_PATH)
print(f"Loaded {len(raw_df)} rows across {raw_df['schema_id'].nunique()} unique schemas")
print(raw_df["complexity"].value_counts())

unique_schema_ids = raw_df["schema_id"].unique()
rng = np.random.default_rng(RNG_SEED)
rng.shuffle(unique_schema_ids)

n = len(unique_schema_ids)
n_val = max(1, int(0.10 * n))
n_heldout = max(1, int(0.10 * n))

val_schema_ids = set(unique_schema_ids[:n_val])
heldout_schema_ids = set(unique_schema_ids[n_val:n_val + n_heldout])
train_schema_ids = set(unique_schema_ids[n_val + n_heldout:])

train_df = raw_df[raw_df["schema_id"].isin(train_schema_ids)].reset_index(drop=True)
val_df = raw_df[raw_df["schema_id"].isin(val_schema_ids)].reset_index(drop=True)
heldout_df = raw_df[raw_df["schema_id"].isin(heldout_schema_ids)].reset_index(drop=True)

print(f"train:   {len(train_df):5d} rows / {len(train_schema_ids):4d} schemas")
print(f"val:     {len(val_df):5d} rows / {len(val_schema_ids):4d} schemas")
print(f"heldout: {len(heldout_df):5d} rows / {len(heldout_schema_ids):4d} schemas")

logger.info(
    f"Schema-grouped split | train={len(train_df)} rows/{len(train_schema_ids)} schemas | "
    f"val={len(val_df)} rows/{len(val_schema_ids)} schemas | "
    f"heldout={len(heldout_df)} rows/{len(heldout_schema_ids)} schemas"
)


Loads the base Qwen3-4B model in 4-bit and attaches a rank-16 LoRA adapter for fine-tuning.
Unchanged from the original notebook — current Unsloth guidance still endorses r=16 (or 32),
alpha=32 (2x rank, also explicitly sanctioned at 1x), dropout=0, and targeting all attention + MLP
projection layers.


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 4096  # generous starting point; revisit after the token-length check below

# bf16 needs an Ampere+ GPU (A100, L4, RTX 30xx+/50xx). Detect at runtime instead of
# hardcoding torch.bfloat16 — a free-tier Colab T4 is Turing-generation and only supports
# fp16; hardcoding bf16 crashes there with "Your setup doesn't support bf16/gpu."
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Using compute dtype: {COMPUTE_DTYPE}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=COMPUTE_DTYPE,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


Just eyeballing one schema string to see what the model will actually be reading.

In [ ]:
train_df["schema"][0]

The system prompt that teaches the model how to read a schema, plus the code that turns each row
into a chat-style training example. Identical to the original notebook's template — the house rules
(relationship direction, `DISTINCT`, avoiding `WITH`-induced accidental grouping, etc.) apply just as
much to a domain-specific schema as to the original public-database ones.


In [ ]:
SYSTEM_PROMPT_TEMPLATE = (
    """You are a Cypher query generation assistant for a Neo4j graph database.

You are given a graph schema and a question in natural language. Use the
schema strictly - it is the only source of truth for what exists in the graph.

How to read the schema:
- 'Node properties' lists each node label together with its properties and
  their types (e.g. STRING, FLOAT, DATE, POINT). Some properties list
  example or available values - these show the kind of data to expect, not
  an exhaustive list to match against literally unless the question refers
  to one of them directly.
- 'The relationships' lists every valid pattern of how node labels connect,
  in the form (:LabelA)-[:REL_TYPE]->(:LabelB). This tells you both the
  relationship type name and its direction - respect the direction when you
  build your MATCH pattern.

How to map the question to the schema:
1. Find the node label(s) the question is really asking about (the subject
  and the target of the question).
2. Find the relationship path in the schema that connects those labels -
  questions often require traversing more than one relationship.
3. Identify any filters mentioned in the question (names, dates, categories,
  thresholds) and match them to the correct property on the correct label.
4. If the question asks for a count, total, average, minimum, maximum, or
  'top N', use the appropriate aggregation function and ORDER BY / LIMIT.

Rules:
- Use only labels, relationship types, and properties that literally appear
  in the schema below. Never invent one.
- Return ONLY the Cypher query - no explanation, no markdown fences, no
  comments.
- Return only the specific properties the question names. Return a whole
  node only when the question asks generally about an entity without naming
  particular attributes.
- When computing a single overall aggregate (an overall average, count, or
  sum), do not carry unrelated variables into the WITH that produces it -
  every non-aggregated variable in a WITH implicitly groups the aggregate by
  that variable, turning one intended overall result into one result per
  group.
- Before returning the query, check every relationship pattern you used against
  the schema's relationship list. Your arrow direction and label order must
  match one of the listed (:LabelA)-[:REL_TYPE]->(:LabelB) patterns exactly -
  if your pattern is the reverse of a listed one, you have the direction
  wrong and must flip it.
- For "highest", "lowest", "top N", "most/least" phrasing, select with
  ORDER BY <property> ASC|DESC LIMIT N rather than computing min()/max() and
  re-matching on equality - re-matching on equality returns every tied row
  instead of one deterministic answer.
- If a MATCH path can reach the same return value multiple times through
  multi-hop or branching traversal, use DISTINCT on it - unless the question
  specifically asks for a count or list per relationship/edge, in which case
  duplicates are the correct answer and DISTINCT must not be used.
- When the question asks about a status, state, count threshold, or yes/no
  condition ("accepted", "active", "at least one", "any", "some", "is X"),
  first check whether the relevant node has a property in the schema that
  directly represents that condition (a BOOLEAN, or a COUNT/INTEGER property
  already tracking it) and filter on it directly. Do not reconstruct the
  condition via a traversal or exists() check if a direct property already
  encodes it.
- If the property the question refers to (e.g. "type", "kind", "category")
  does not exist on the node you first match, do not traverse further away
  from it searching for a substitute property on a different node. Stay on
  the matched node and use its closest literal property (e.g. count distinct
  values of an existing identifying property on that same node) rather than
  inventing a multi-hop path to a loosely related property elsewhere.
- Return ONLY the Cypher query - no explanation, no markdown fences, no
  comments.\n\nSchema:\n{schema}"""
)

def build_messages(row):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_TEMPLATE.format(schema=row["schema"])},
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["cypher"]},
        ]
    }

train_records = [build_messages(row) for _, row in train_df.iterrows()]

pprint(train_records[0])


Builds the same kind of chat-style records for the **validation** split — these feed `eval_loss`
during training, so `Trainer` can tell if it's overfitting instead of finding out only after the
fact.


In [ ]:
val_records = [build_messages(row) for _, row in val_df.iterrows()]
print(f"{len(val_records)} validation records")


Wraps the tokenizer's chat template so we can render a conversation to plain text or straight to
token ids.


In [ ]:
def apply_template(messages, tokenize, add_generation_prompt, return_tensors=None):
    kwargs = dict(
        tokenize=tokenize,
        add_generation_prompt=add_generation_prompt,
    )
    if return_tensors:
        kwargs["return_tensors"] = return_tensors

    return tokenizer.apply_chat_template(
        messages,
        enable_thinking=False,
        **kwargs
    )

def render_chat(messages):
    text = apply_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    text = text.replace("<think>\n\n</think>\n", "")
    return text


Sanity check — what does one fully rendered training example actually look like?

In [ ]:
print(render_chat(train_records[0]["messages"])[:])

Checks how long the rendered examples really are, so `MAX_SEQ_LENGTH` isn't set blindly. Run over
both train and validation records — a domain-specific schema may be shorter or longer than the
original dataset's, and this is a fresh check, not an assumption carried over.


In [ ]:
all_records_for_length_check = train_records + val_records
lengths = np.array([
    len(
        tokenizer(
            render_chat(r["messages"]),
            add_special_tokens=False,
        )["input_ids"]
    )
    for r in all_records_for_length_check
])

for p in [50, 75, 90, 95, 99]:
    print(f"P{p}: {int(np.percentile(lengths, p))}")

print("max:", lengths.max())
print(
    f"Samples that would be truncated at {MAX_SEQ_LENGTH}: "
    f"{(lengths > MAX_SEQ_LENGTH).sum()} / {len(lengths)}"
)


Tokenizes each example and masks out the prompt tokens so loss is only computed on the
assistant's answer. Unchanged mechanically from the original notebook.


In [ ]:
ASSISTANT_MARKER = "<|im_start|>assistant\n"

def tokenize_and_mask(record):
    full_text = render_chat(record["messages"])

    input_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )["input_ids"]

    marker_pos = full_text.rfind(ASSISTANT_MARKER)
    assert marker_pos != -1, "Assistant marker not found."

    prompt_text = full_text[: marker_pos + len(ASSISTANT_MARKER)]

    prompt_len = len(
        tokenizer(
            prompt_text,
            add_special_tokens=False,
        )["input_ids"]
    )

    prompt_len = min(prompt_len, len(input_ids))

    labels = input_ids.copy()
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


Runs that tokenization over both the train and validation records and wraps each as a HF
`Dataset`. Two datasets now instead of one — `val_dataset` is new, and it's what `eval_dataset` in
`TrainingArguments` will point at.


In [ ]:
def build_dataset(records):
    tokenized = []

    for r in records:
        t = tokenize_and_mask(r)
        tokenized.append(t)

    return Dataset.from_list(tokenized)

train_dataset = build_dataset(train_records)
val_dataset = build_dataset(val_records)

print(train_dataset)
print(val_dataset)


Custom collator that pads each batch to its longest example — we're not using the tokenizer's
default one. Works unchanged for both the training loop and the evaluation loop, since it always
builds `labels` for every row either way.


In [ ]:
def collate_fn(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_len = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_id] * pad_len)
        attention_mask.append(ex["attention_mask"] + [0] * pad_len)
        labels.append(ex["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


Just confirming which tokenizer actually got loaded.

In [ ]:
tokenizer.name_or_path

Training hyperparameters — this is where most of the research-driven changes land.

- **Effective batch size 16** (`per_device_train_batch_size=4` x `gradient_accumulation_steps=4`),
  down from 128 — LoRA handles large batches worse than full fine-tuning, and a batch this size on
  ~3,200 training rows gives roughly 200 steps/epoch instead of the ~7 steps/epoch a batch of 128
  would have given.
- **Learning rate 1.5e-4**, up from 2e-5 — LoRA's optimal LR runs roughly 10-15x higher than full-FT,
  especially for short runs.
- **3 epochs, but gated by early stopping** — `load_best_model_at_end=True` +
  `metric_for_best_model="eval_loss"` mean the actual stopping point is decided by validation loss,
  not a fixed guess.
- **`eval_steps` and `save_steps` are set to the same value on purpose** — if they differ, early
  stopping silently does nothing useful until a checkpoint happens to coincide with an eval step.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints_text2cypher"),

    # This training recipe's originally-intended batch sizes — safe on a real ~32GB GPU
    # (e.g. RTX 5090). The Colab copy of this notebook had to shrink these (and add
    # eval_accumulation_steps) to survive a free-tier T4's 14.5GB, where the eval loop's
    # fp32 logits conversion (batch * seq_len * vocab_size * 4 bytes) OOMs above
    # per_device_train_batch_size=2 / per_device_eval_batch_size=2. With 32GB of headroom
    # that ceiling is far higher — if you still hit an OOM here, that Colab fix (smaller
    # batches + eval_accumulation_steps=1) is the same lever to pull.
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    # Bumped from 3 -> 5. The v1 Colab run's EarlyStoppingCallback fired at step 240/405
    # (~1.3 epochs) because eval_loss plateaued around 0.174-0.180 for several consecutive
    # checkpoints on a small (216-270 row) validation set — that's a real plateau on this
    # recipe, not just noise, but patience=3 (only ~60 steps / <0.5 epoch of tolerance) cut
    # it off before the extra epoch budget even mattered. More epochs alone don't help if
    # patience is still tight enough to stop before using them, hence bumping both together.
    num_train_epochs=5,

    learning_rate=1.5e-4,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_torch_fused",

    bf16=COMPUTE_DTYPE == torch.bfloat16,
    fp16=COMPUTE_DTYPE == torch.float16,

    logging_steps=10,
    logging_strategy="steps",
    disable_tqdm=False,

    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=42,
    report_to="none",
    push_to_hub=False,
    remove_unused_columns=False,
)


A `TrainerCallback` that mirrors every training *and* evaluation log line (loss, eval_loss,
learning rate, step, epoch) into the run's log file — unchanged from the original notebook, it
already handles eval-loss logging with no modification since `on_log` fires for both loops.


In [ ]:
import time
from transformers import TrainerCallback

class FileLoggingCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        logger.info(
            f"Training started | epochs={args.num_train_epochs} | "
            f"per_device_batch={args.per_device_train_batch_size} | "
            f"grad_accum={args.gradient_accumulation_steps} | "
            f"effective_batch={args.per_device_train_batch_size * args.gradient_accumulation_steps} | "
            f"lr={args.learning_rate} | lr_schedule={args.lr_scheduler_type} | seed={args.seed}"
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            metrics = " | ".join(f"{k}={v}" for k, v in logs.items())
            logger.info(f"step={state.global_step}/{state.max_steps} | {metrics}")

    def on_train_end(self, args, state, control, **kwargs):
        duration = time.time() - self.train_start
        logger.info(
            f"Training finished | duration={duration/60:.1f} min | "
            f"final_step={state.global_step} | epochs_completed={state.epoch:.2f}"
        )


Kicks off the actual LoRA fine-tuning run. Two additions versus the original notebook:
`eval_dataset=val_dataset` (so `eval_loss` actually gets computed), and an `EarlyStoppingCallback`
that stops training once validation loss stops improving for 3 consecutive eval checkpoints, instead
of always running the full epoch count regardless of what's happening.


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[
        FileLoggingCallback(),
        # Bumped from 3 -> 6 (120 steps / ~0.9 epoch of tolerance instead of ~60 steps /
        # <0.5 epoch). The v1 Colab run stopped at patience=3 during a real eval_loss
        # plateau on a small, noisy validation set (216-270 rows) — a temporary flat
        # stretch shouldn't end the run before the extra epoch budget above gets used.
        EarlyStoppingCallback(early_stopping_patience=6),
    ],
)

try:
    train_result = trainer.train()
    logger.info(f"train_result: {train_result.metrics}")
except Exception:
    logger.exception("Training failed")
    raise


Saves the LoRA adapter and tokenizer locally.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "qwen3_4b_cypher_lora"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))


Merges the adapter into the base weights and pushes the full model to the Hub — needs a real HF
write token, entered here at runtime via `getpass` so it's never written into this notebook file.
Skip this cell entirely if you don't want to publish the model. `HF_REPO_ID` defaults to a name
distinct from the original run's model, so it doesn't overwrite it.


In [ ]:
from getpass import getpass

HF_REPO_ID = "your-username/text2cypher-lora"  # edit to your own namespace before pushing
HF_TOKEN = getpass("Hugging Face write token (hidden input): ")

model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)


Runs the fine-tuned model on a single schema/question pair and returns the Cypher it generates.
Unchanged — the message construction here must stay in sync with `SYSTEM_PROMPT_TEMPLATE` and
`build_messages` above, since train and inference prompts have to match exactly.


In [ ]:
import warnings
warnings.simplefilter("ignore")

def generate_cypher(schema, question, max_new_tokens=400):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT_TEMPLATE.format(schema=schema)
        },
        {
            "role": "user",
            "content": question
        },
    ]

    input_ids = apply_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    generated = output[0][input_ids.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


Wraps `generate_cypher` with a post-hoc schema-grounding check — the exact same
`check_grounding` validator from `validate_and_build.py` that every training row had to pass
before it was allowed into the dataset. Real eval on the trained model (see
prior eval runs) found the model's actual semantic
accuracy is much higher than raw exact-match suggests (~60%, vs. an ~8% exact-match that's
mostly punishing cosmetic variable-naming differences) — but a meaningful minority of
failures are genuine reasoning errors: dropped joins/filters, extra unnecessary hops, or a
misattributed property. None of those observed failures were outright hallucinated schema
elements (the model stayed grounded), but there's no guarantee that holds on unseen production
schemas, so this check stays as a real safety net rather than an assumption. If the first
generation references a label/relationship/property not declared in the schema, this retries
once with an explicit correction hint; if it still fails, it returns `None` plus the specific
grounding violations instead of silently handing a caller a broken query.


In [ ]:
import re

MIDDOT = "\u00b7"

def _parse_schema_for_grounding(schema_text):
    """Self-contained copy of validate_and_build.py's parse_schema/check_grounding logic --
    duplicated here (rather than imported) so this cell works standalone even if
    validate_and_build.py isn't present on the Colab Drive folder, which only holds the
    dataset by default."""
    labels = set(re.findall(r"\(:(\w+)\)", schema_text))
    rel_types = set(re.findall(r"\[:(\w+)\]", schema_text))
    all_props = set()
    for line in schema_text.splitlines():
        m = re.match(rf"\s*{MIDDOT}\s*(\w+)\s*:", line)
        if m:
            all_props.add(m.group(1))
    return {"labels": labels, "rel_types": rel_types, "all_props": all_props}


def check_grounding(schema_text, cypher):
    parsed = _parse_schema_for_grounding(schema_text)
    reasons = []
    used_labels = set(re.findall(r":(\w+)\s*[\{\)]", cypher)) - parsed["rel_types"]
    used_rel_types = set(re.findall(r"\[:(\w+)\]", cypher))
    used_props = set(re.findall(r"(?<!\d)\.(\w+)\b", cypher)) | set(re.findall(r"\{\s*(\w+)\s*:", cypher))
    used_props = {p for p in used_props if not p.isdigit()}

    bad_labels = used_labels - parsed["labels"]
    bad_rels = used_rel_types - parsed["rel_types"]
    bad_props = used_props - parsed["all_props"]
    if bad_labels:
        reasons.append(f"undeclared labels: {sorted(bad_labels)}")
    if bad_rels:
        reasons.append(f"undeclared relationship types: {sorted(bad_rels)}")
    if bad_props:
        reasons.append(f"undeclared properties: {sorted(bad_props)}")
    return (len(reasons) == 0, reasons)


def generate_cypher_checked(schema, question, max_new_tokens=400, max_retries=1):
    cypher = generate_cypher(schema, question, max_new_tokens=max_new_tokens)
    ok, reasons = check_grounding(schema, cypher)
    if ok:
        return cypher, True, []

    if max_retries > 0:
        correction_hint = (
            f"\n\n(Your previous answer referenced something not declared in the schema "
            f"above -- {'; '.join(reasons)}. Only use labels, relationship types, and "
            f"properties explicitly listed in the schema. Try again.)"
        )
        cypher_retry = generate_cypher(schema, question + correction_hint, max_new_tokens=max_new_tokens)
        ok_retry, reasons_retry = check_grounding(schema, cypher_retry)
        if ok_retry:
            return cypher_retry, True, []
        return cypher_retry, False, reasons_retry

    return cypher, False, reasons


Scores the model against ground truth with a strict **string exact-match** (after normalizing
whitespace and a trailing `;`) — this is a lower bound on real accuracy, since two Cypher queries
that are worded differently (clause order, quoting, aliasing) can still return identical results.


In [ ]:
import re

def normalize_cypher(query):
    q = re.sub(r"\s+", " ", query.strip())
    return q.rstrip(";").strip()

def is_exact_match(pred, gold):
    return normalize_cypher(pred) == normalize_cypher(gold)


Classifies which schema-string format a row uses. If the synthetic dataset was generated in a
single standardized format, every row should land in one bucket here — this cell becomes a sanity
check that formatting actually stayed consistent, rather than a meaningful breakdown axis like it was
for the original mixed-format dataset.


In [ ]:
def classify_schema_format(schema):
    s = schema.strip()
    if s.startswith("Node properties:"):
        return "markdown_bullets"
    if s.startswith("Node properties are the following"):
        return "quoted_sentence"
    if s.startswith("Graph schema: Relevant node labels"):
        return "graph_schema_prose"
    if s.startswith("Relevant node labels and their properties"):
        return "relevant_labels_prose"
    if s.startswith("{"):
        return "raw_json"
    return "other"


Runs the model over a dataframe of rows (`schema`/`question`/`cypher`/`complexity`/`domain`) and
records whether each prediction exact-matches the gold query, plus per-row latency and schema
format. Pass `n` to sample a subset first for a quick check before committing to a full pass.


In [ ]:
from tqdm.auto import tqdm

def evaluate(df, n=None, seed=42, name="eval"):
    eval_df = df if n is None else df.sample(n=min(n, len(df)), random_state=seed)
    logger.info(f"[{name}] Starting evaluation on {len(eval_df)} rows")
    start = time.time()

    rows = []
    n_errors = 0
    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        t0 = time.perf_counter()
        error = None
        try:
            pred = generate_cypher(schema=row["schema"], question=row["question"])
        except Exception as e:
            pred = ""
            error = str(e)
            n_errors += 1
            logger.error(f"[{name}] row={idx} generation failed: {e}")
        latency = time.perf_counter() - t0

        rows.append({
            "row_index": idx,
            "question": row["question"],
            "gold": row["cypher"],
            "predicted": pred,
            "complexity": row["complexity"],
            "domain": row.get("domain"),
            "schema_format": classify_schema_format(row["schema"]),
            "exact_match": is_exact_match(pred, row["cypher"]) if error is None else False,
            "latency_sec": latency,
            "error": error,
        })

    duration = time.time() - start
    avg_latency = sum(r["latency_sec"] for r in rows) / len(rows)
    logger.info(
        f"[{name}] Finished in {duration/60:.1f} min | {n_errors} errors | "
        f"avg latency {avg_latency:.2f}s/row"
    )
    return pd.DataFrame(rows)


Aggregates exact-match accuracy overall, by `complexity`, by `domain`, and by schema text
format — the actual performance report. Also saves the full row-by-row results to CSV.


In [ ]:
def score_report(results_df, name="eval", save_csv=True):
    overall = results_df["exact_match"].mean()
    n = len(results_df)
    summary = f"[{name}] {overall:.1%} exact-match ({int(results_df['exact_match'].sum())}/{n})"
    print(f"=== {summary} ===\n")
    logger.info(summary)

    by_complexity = (
        results_df.groupby("complexity")["exact_match"]
        .agg(accuracy="mean", n="count")
        .sort_values("n", ascending=False)
    )
    print("By complexity:")
    print(by_complexity)
    logger.info(f"[{name}] by complexity: {by_complexity.to_dict('index')}")

    by_domain = (
        results_df.groupby("domain")["exact_match"]
        .agg(accuracy="mean", n="count")
        .sort_values("n", ascending=False)
    )
    print("\nBy domain:")
    print(by_domain)
    logger.info(f"[{name}] by domain: {by_domain.to_dict('index')}")

    by_schema_format = (
        results_df.groupby("schema_format")["exact_match"]
        .agg(accuracy="mean", n="count")
        .sort_values("n", ascending=False)
    )
    print("\nBy schema text format:")
    print(by_schema_format)
    logger.info(f"[{name}] by schema format: {by_schema_format.to_dict('index')}")

    if save_csv:
        path = EVAL_DIR / f"eval_results_{name}.csv"
        path.parent.mkdir(parents=True, exist_ok=True)
        results_df.to_csv(path, index=False)
        print(f"\nFull row-by-row results saved to {path}")
        logger.info(f"[{name}] row-level results saved to {path}")

    return overall


Runs the full generation-based scoring pass on the **validation** split. Worth being explicit
about what this number means: validation was already used to pick the best checkpoint
(`load_best_model_at_end`), so this accuracy is mildly optimistic — a real (if weak) form of
information leakage, just far less than training on it directly. Treat this as a sanity check that
low eval_loss actually corresponds to correct Cypher, not as the headline number.


In [ ]:
val_eval_df = val_df[["schema", "question", "cypher", "complexity", "domain"]].reset_index(drop=True)

val_results = evaluate(val_eval_df, name="validation")
score_report(val_results, name="validation")


Runs the same scoring pass on the **heldout** split — schemas the model never saw during
training *or* during validation-based checkpoint selection. This is the real generalization number,
the one that actually predicts how this model will behave on an entirely new schema.


In [ ]:
heldout_eval_df = heldout_df[["schema", "question", "cypher", "complexity", "domain"]].reset_index(drop=True)

heldout_results = evaluate(heldout_eval_df, name="heldout")
score_report(heldout_results, name="heldout")


Assembles everything collected above — environment, model/LoRA config, dataset sizes (including
the schema-grouped split sizes), the training loss/eval-loss history, and both benchmark sections —
into one report. Writes `.txt` and `.md` unconditionally, and a `.pdf` if `reportlab` is installed.


In [ ]:
def _g(name, default=None):
    return globals().get(name, default)

def _benchmark_section(name, results_df):
    if results_df is None:
        return [f"-- BENCHMARK: {name} --", "Not run in this session.", ""]

    n = len(results_df)
    acc = results_df["exact_match"].mean()
    n_errors = int(results_df["error"].notna().sum())

    lines = [
        f"-- BENCHMARK: {name} --",
        f"Rows evaluated: {n}",
        f"Overall exact-match accuracy: {acc:.1%} ({int(results_df['exact_match'].sum())}/{n})",
        f"Generation errors: {n_errors}",
        f"Latency (seconds/row) — mean: {results_df['latency_sec'].mean():.2f} | "
        f"p50: {results_df['latency_sec'].median():.2f} | "
        f"p95: {results_df['latency_sec'].quantile(0.95):.2f}",
        "",
        "By complexity:",
        results_df.groupby("complexity")["exact_match"].agg(accuracy="mean", n="count").to_string(),
        "",
        "By domain:",
        results_df.groupby("domain")["exact_match"]
            .agg(accuracy="mean", n="count").sort_values("n", ascending=False).to_string(),
        "",
        "By schema text format:",
        results_df.groupby("schema_format")["exact_match"]
            .agg(accuracy="mean", n="count").sort_values("n", ascending=False).to_string(),
        "",
        "Worst-case examples (first 5 mismatches):",
    ]

    mismatches = results_df[~results_df["exact_match"]].head(5)
    for _, r in mismatches.iterrows():
        lines.append(f"  Q:    {r['question']}")
        lines.append(f"  GOLD: {r['gold']}")
        lines.append(f"  PRED: {r['predicted']}")
        if r.get("error"):
            lines.append(f"  ERROR: {r['error']}")
        lines.append("")

    lines.append("")
    return lines

def build_report_text():
    lines = []
    lines.append("=" * 70)
    lines.append("TEXT2CYPHER LORA FINE-TUNE — PERFORMANCE REPORT")
    lines.append("=" * 70)
    lines.append(f"Run ID: {RUN_ID}")
    lines.append(f"Generated: {datetime.now().isoformat()}")
    lines.append(f"Log file: {LOG_PATH}")
    lines.append("")

    lines.append("-- ENVIRONMENT --")
    lines.append(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        lines.append(f"GPU: {torch.cuda.get_device_name(0)}")
        lines.append(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    lines.append(f"PyTorch: {torch.__version__}")
    lines.append("")

    lines.append("-- MODEL / LoRA CONFIG --")
    lines.append(f"Base model: {_g('MODEL_NAME', 'unknown')}")
    lines.append(f"Max sequence length: {_g('MAX_SEQ_LENGTH', 'unknown')}")
    model_obj = _g("model")
    if model_obj is not None and hasattr(model_obj, "get_nb_trainable_parameters"):
        trainable, total = model_obj.get_nb_trainable_parameters()
        lines.append(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    else:
        lines.append("Trainable parameter count: unavailable (model not loaded in this session)")
    lines.append("")

    lines.append("-- DATASET (schema-grouped split) --")
    raw_df_obj = _g("raw_df")
    if raw_df_obj is not None:
        lines.append(f"Total rows: {len(raw_df_obj):,} across {raw_df_obj['schema_id'].nunique():,} unique schemas")
    train_df_obj = _g("train_df")
    if train_df_obj is not None:
        lines.append(f"Train rows: {len(train_df_obj):,} ({train_df_obj['schema_id'].nunique():,} schemas)")
        lines.append(f"Train complexity distribution: {train_df_obj['complexity'].value_counts().to_dict()}")
    val_df_obj = _g("val_df")
    if val_df_obj is not None:
        lines.append(f"Validation rows: {len(val_df_obj):,} ({val_df_obj['schema_id'].nunique():,} schemas, used for checkpoint selection)")
    heldout_df_obj = _g("heldout_df")
    if heldout_df_obj is not None:
        lines.append(f"Heldout rows: {len(heldout_df_obj):,} ({heldout_df_obj['schema_id'].nunique():,} schemas, never touched during training)")
    lines.append("")

    lines.append("-- TRAINING --")
    training_args_obj = _g("training_args")
    if training_args_obj is not None:
        lines.append(f"Epochs (max): {training_args_obj.num_train_epochs}")
        lines.append(f"Per-device train batch size: {training_args_obj.per_device_train_batch_size}")
        lines.append(f"Gradient accumulation steps: {training_args_obj.gradient_accumulation_steps}")
        lines.append(
            f"Effective batch size: "
            f"{training_args_obj.per_device_train_batch_size * training_args_obj.gradient_accumulation_steps}"
        )
        lines.append(f"Learning rate: {training_args_obj.learning_rate}")
        lines.append(f"LR schedule: {training_args_obj.lr_scheduler_type}")
        lines.append(f"Early stopping: enabled (patience=3, metric=eval_loss)")
        lines.append(f"Seed: {training_args_obj.seed}")
    else:
        lines.append("Training args unavailable (training cell not run this session)")

    trainer_obj = _g("trainer")
    if trainer_obj is not None:
        history = trainer_obj.state.log_history
        losses = [(h["step"], h["loss"]) for h in history if "loss" in h]
        eval_losses = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
        if losses:
            lines.append(f"Logged train-loss points: {len(losses)}")
            lines.append(f"First: step={losses[0][0]} loss={losses[0][1]:.4f}")
            lines.append(f"Last:  step={losses[-1][0]} loss={losses[-1][1]:.4f}")
            lines.append(f"Min train loss observed: {min(l for _, l in losses):.4f}")
        if eval_losses:
            lines.append(f"Logged eval-loss points: {len(eval_losses)}")
            lines.append(f"Best eval loss: {min(l for _, l in eval_losses):.4f} at step {min(eval_losses, key=lambda x: x[1])[0]}")
    else:
        lines.append("Training loss history unavailable (trainer not run this session)")
    lines.append("")

    lines.append(
        "NOTE: the validation-set benchmark below was used to pick the best checkpoint "
        "(load_best_model_at_end), so it is mildly optimistic. The heldout-set benchmark is the "
        "trustworthy generalization number — those schemas were never touched during training or "
        "checkpoint selection."
    )
    lines.append("")

    lines.extend(_benchmark_section("VALIDATION SET (used for checkpoint selection)", _g("val_results")))
    lines.extend(_benchmark_section("HELDOUT SET (never touched during training)", _g("heldout_results")))

    lines.append("-- ARTIFACTS --")
    lines.append(f"Full log file: {LOG_PATH}")
    if _g("val_results") is not None:
        lines.append("Row-level validation results: eval_results_validation.csv")
    if _g("heldout_results") is not None:
        lines.append("Row-level heldout results: eval_results_heldout.csv")

    return "\n".join(lines)


Writes the report to `outputs/logs/report_<RUN_ID>.txt` and `.md` (always), and `.pdf` too if
`reportlab` is installed.


In [ ]:
report_text = build_report_text()

report_txt_path = LOG_DIR / f"report_{RUN_ID}.txt"
report_md_path = LOG_DIR / f"report_{RUN_ID}.md"
report_txt_path.write_text(report_text)
report_md_path.write_text(report_text)
logger.info(f"Report written to {report_txt_path} and {report_md_path}")

def export_report_pdf(text, path):
    try:
        from reportlab.lib.pagesizes import LETTER
        from reportlab.pdfgen import canvas
    except ImportError:
        return False

    c = canvas.Canvas(str(path), pagesize=LETTER)
    width, height = LETTER
    x, y = 0.6 * 72, height - 0.6 * 72
    line_height = 11

    c.setFont("Courier", 8)
    for line in text.split("\n"):
        if y < 0.6 * 72:
            c.showPage()
            c.setFont("Courier", 8)
            y = height - 0.6 * 72
        c.drawString(x, y, line[:120])
        y -= line_height
    c.save()
    return True

report_pdf_path = LOG_DIR / f"report_{RUN_ID}.pdf"
if export_report_pdf(report_text, report_pdf_path):
    logger.info(f"PDF report written to {report_pdf_path}")
    print(f"Report written to: {report_txt_path}, {report_md_path}, {report_pdf_path}")
else:
    logger.warning("reportlab not installed — skipped PDF export (.txt/.md were still written).")
    print(f"Report written to: {report_txt_path}, {report_md_path}")
    print("PDF skipped — run `pip install reportlab` and re-run this cell if you want a PDF too.")

print("\n" + report_text)
